# Starionary non-linear control problems
In this notebook, we explain how to solve a stationary non-linear control problems with the package control.

## A stationary non-linear Poisson control problem
As first example, we consider the following stationary non-linear Poisson control problem:

$$
\min_{v,u} \frac{1}{2} \| v - v_d \| ^2 + \frac{1}{\beta} \| u \| ^2$$

subject to:

$$
    - \nabla \cdot(v^2 \; \nabla v) = f + u \qquad \mathrm{in} \; \Omega := (-1, 1)^2, 
$$

with $v = 0$ on $\partial \Omega$, where we set $\beta=10^{-3}$, $v_d = \cos(\frac{\pi x_1}{2}) \cos(\frac{\pi x_2}{2})$, and $f = 0$.

We consider a uniform grid with 10 points in each spatial direction, and employ $P_1$ finite elements as the spatial discretization.

In order to define the problem, we first import the modules from the control class.

In [ ]:
from firedrake import *
from control.control import *

We now defined the geometry of the problem and the function space where we seek the solution.

In [ ]:
mesh = RectangleMesh(10, 10, 1.0, 1.0, originX=-1.0, originY=-1.0)
space_0 = FunctionSpace(mesh, "Lagrange", 1)

Then, we define the desired state $v_d$, the force function $f$, and the boundary conditions to apply to $v$.

In [ ]:
# the desired state
def desired_state(test):
    space = test.function_space()
    mesh = space.mesh()
    X = SpatialCoordinate(mesh)
    x_1 = X[0]
    x_2 = X[1]

    v_d = Function(space, name="v_d")
    v_d.interpolate(cos(0.5 * pi * x_1) * cos(0.5 * pi * x_2))

    return inner(v_d, test) * dx, v_d


# the force function
def force_f(test):
    space = test.function_space()

    f = Function(space, name="f")
    f.zero()

    return inner(f, test) * dx


# the boundary conditions
bcs_v = DirichletBC(space_0, 0.0, "on_boundary")

Finally, we define the form that represents the differential operator in space. This has to be passed as the bilinear form representing the Picard linearization, as follows.

In [ ]:
# the forward form
def forw_diff_operator(trial, test, v):
    return inner((v**2.0) * grad(trial), grad(test)) * dx

Now, we can define the Poisson control problem to be solved. Note that since for the problem above we have $f=0$ we can avoid passing the kwarg force_function to the constructor, as default the option is assuming zero force function (similarly for the kwarg desired_state in case $v_d=0$). 

In [ ]:
non_linear_Poisson_control = Control.Stationary(
    space_0, forw_diff_operator, desired_state=desired_state,
    force_function=force_f, bcs_v=bcs_v, Gauss_Newton=True)

With the setting above, the non-linear solver is adopting a Gauss-Newton linearization of the non-linear problem, set using Gauss_Newton=True. Gauss_Newton=False would select a Picard linearization.

Non-linear control problems are solved by calling the module non_linear_solve(). Default options run the solver for 10 non-linear iterations until the absolute non-linear residual is reduced by $10^{-8}$ or the relative non-linear residual is reduced by $10^{-5}$. In order to modify the options, one has to pass specific kwargs to the call. In addition, one can pass the options of the solver for the system derived from the linearization adopted, together with the options for the inner solvers.

For example, suppose we would like to run Gauss-Newton for 20 iterations, seeking for a reduction of $10^{-5}$ on both the absolute and relative non-linear residual, and we would like to employ 500 iterations of preconditioned FGMRES as linear solver, with restart every 10 iterations, and apply the in-built preconditioner, with Chebyshev semi-iteration for approximating the inverse of the (1,1)-block. Then, we pass the following options to the module.

In [ ]:
solver_parameters = {"linear_solver": "fgmres",
                     "maximum_iterations": 500,
                     "fgmres_restart": 10,
                     "relative_tolerance": 1.0e-6,
                     "absolute_tolerance": 1.0e-6,
                     "monitor_convergence": True}

# bounds on the eigenvalues for the preconditioned mass matrix
e_min_p = 0.5
e_max_p = 2.0

sp_11block = {"ksp_type": "chebyshev",
              "pc_type": "jacobi",
              "ksp_chebyshev_eigenvalues": f"{e_min_p:.16e}, {e_max_p:.16e}",
              "ksp_chebyshev_esteig": "0.0,0.0,0.0,0.0",
              "ksp_chebyshev_esteig_steps": 0,
              "ksp_chebyshev_esteig_noisy": False,
              "ksp_max_it": 20,
              "ksp_atol": 0.0,
              "ksp_rtol": 0.0}

auxiliary_sp = {"sp_11block": sp_11block}

non_linear_Poisson_control.non_linear_solve(
    solver_parameters=solver_parameters, auxiliary_sp=auxiliary_sp,
    max_non_linear_iter=20, absolute_non_linear_tol=1.0e-5,
    relative_non_linear_tol=1.0e-5,
    create_output=False, plots=False)

## Non-linear incompressible control problems
We now consider the case of incompressible control problems. We consider the following stationary incompressible Navier–Stokes control as an example:

$$
\min_{\vec{v},\vec{u}} \frac{1}{2} \| \vec{v} - \vec{v}_d \| ^2 + \frac{1}{\beta} \| \vec{u} \| ^2
$$

subject to:

$$
    - \nu \nabla^2 \; \vec{v} + \vec{v}\cdot \nabla \vec{v} + \nabla p = \vec{f} + \vec{u} \qquad \mathrm{in} \; \Omega := (-1, 1)^2, 
$$
$$
    - \nabla \cdot \vec{v} = 0 \qquad \mathrm{in} \; \Omega,
$$
$$
    \vec{v} = \vec{g} \qquad \mathrm{on} \; \partial \Omega.
$$

We consider the lid-driven cavity problem, with force function $\vec{f}=[0, 0]^\top$ and boundary conditions given by

$$
    \vec{g} = [1, 0]^\top \quad \mathrm{on} \; \partial \Omega_1:=(-1,1) \times \{ 1\},
$$
$$
    \vec{g} = [0, 0]^\top \quad \mathrm{on} \; \partial \Omega \setminus \partial \Omega_1.
$$

We seek $\vec{v}_d = [0, 0]^\top$ as desired state, with viscosity $\nu = \frac{1}{100}$, and set $\beta = 10^{-4}$.

As we have done above, we first import all the important modules, build the mesh, define the function spaces where seeking the solution together with the boundary conditions for the velocity, and define the force function and the desired state. Then, we can call the constructor of the Stationary class. For this problem, we consider $\mathbf{P}_2$–$\mathbf{P}_1$ finite element pair. As mentioned in the Stokes control problem, the pressure space can be passed directly to the non-linear solver, as we do so here.

In [ ]:
from firedrake import *
from control.preconditioner import ConstantNullspace
from control.control import *

beta = 1.0e-4

mesh = RectangleMesh(10, 10, 1.0, 1.0, originX=-1.0, originY=-1.0)

space_v = VectorFunctionSpace(mesh, "Lagrange", 2)
space_p = FunctionSpace(mesh, "Lagrange", 1)

# the boundary conditions
bcs_v = [DirichletBC(space_v, Constant((1.0, 0.0)), (4,)),
         DirichletBC(space_v, 0.0, (1, 2, 3))]


# the forward form
def forw_diff_operator(trial, test, u):
    # viscosity
    nu = 1.0 / 100.0
    # spatial differential for the forward problem
    return (
        nu * inner(grad(trial), grad(test)) * dx
        + inner(dot(grad(trial), u), test) * dx)


# the desired state
def desired_state(test):
    space = test.function_space()

    # desired state
    v_d = Function(space, name="v_d")
    v_d.zero()

    return inner(v_d, test) * dx, v_d


# the force function
def force_f(test):
    space = test.function_space()

    # force function
    f = Function(space)
    f.zero()

    return inner(f, test) * dx


stationart_Navier_Stokes_control = Control.Stationary(
    space_v, forw_diff_operator, desired_state=desired_state,
    force_function=force_f, beta=beta, bcs_v=bcs_v)

We employ a Picard linearization of the optimality conditions, running 10 Picard iterations and seeking a reduction of $10^{-5}$ on the relative non-linear residual. At each non-linear iteration, we employ the in-built solver, that run preconditioned FGMRES with restart every 10 iterations, and apply the matching strategy for solving for the (1,1)-block and the block-pressure convection–diffusion preconditioner for approximating the Schur complement. We employ 20 Chebyshev semi-iterations for approximating the inverse of the vector- and pressure mass matrices. In order to run the non-linear solver, we have to call the module incompressible_non_linear_solve(), to whom the user has to provide as an input the nullspace of the corresponding forward stationary Navier–Stokes problem.

In [ ]:
# employing Chebyshev for the (1,1)-block
e_min_v = 0.3924
e_max_v = 2.0598
sp_11block = {
    "ksp_type": "chebyshev",
    "pc_type": "jacobi",
    "ksp_chebyshev_eigenvalues": f"{e_min_v:.16e}, {e_max_v:.16e}",
    "ksp_chebyshev_esteig": "0.0,0.0,0.0,0.0",
    "ksp_chebyshev_esteig_steps": 0,
    "ksp_chebyshev_esteig_noisy": False,
    "ksp_max_it": 20,
    "ksp_atol": 0.0,
    "ksp_rtol": 0.0}

# employing Chebyshev for the pressure-mass matrix
e_min_p = 0.5
e_max_p = 2.0
sp_M_p = {
    "ksp_type": "chebyshev",
    "pc_type": "jacobi",
    "ksp_chebyshev_eigenvalues": f"{e_min_p:.16e}, {e_max_p:.16e}",
    "ksp_chebyshev_esteig": "0.0,0.0,0.0,0.0",
    "ksp_chebyshev_esteig_steps": 0,
    "ksp_chebyshev_esteig_noisy": False,
    "ksp_max_it": 20,
    "ksp_atol": 0.0,
    "ksp_rtol": 0.0}

auxiliary_sp = {"sp_11block": sp_11block,
                "sp_M_p": sp_M_p}

stationart_Navier_Stokes_control.incompressible_non_linear_solve(
    ConstantNullspace(), space_p=space_p, auxiliary_sp=auxiliary_sp,
    max_non_linear_iter=10, relative_non_linear_tol=1.0e-5,
    print_error_non_linear=True, create_output=False, plots=False)